<a href="https://colab.research.google.com/github/SPARSHTHALYARI/SPARSHTHALYARI/blob/main/Customer_ChurnDetectionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading And Describing The Data

In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from datetime import datetime  # Importing datetime module

# Load the dataset
data = pd.read_csv('/content/customer_churn_dataset-training-master.csv')

# Display basic statistics of the dataset
print(data.describe())


          CustomerID            Age         Tenure  Usage Frequency  \
count  440832.000000  440832.000000  440832.000000    440832.000000   
mean   225398.667955      39.373153      31.256336        15.807494   
std    129531.918550      12.442369      17.255727         8.586242   
min         2.000000      18.000000       1.000000         1.000000   
25%    113621.750000      29.000000      16.000000         9.000000   
50%    226125.500000      39.000000      32.000000        16.000000   
75%    337739.250000      48.000000      46.000000        23.000000   
max    449999.000000      65.000000      60.000000        30.000000   

       Support Calls  Payment Delay    Total Spend  Last Interaction  \
count  440832.000000  440832.000000  440832.000000     440832.000000   
mean        3.604437      12.965722     631.616223         14.480868   
std         3.070218       8.258063     240.803001          8.596208   
min         0.000000       0.000000     100.000000          1.000000   


# Data Cleaning and Encoding Categorial Variable
Disclaimer-We have Used Cleaned Dataset


In [21]:
# Apply one-hot encoding to specific categorical columns
data_encoded = pd.get_dummies(data, columns=['Gender', 'Subscription Type', 'Contract Length'], drop_first=True)

# Check the first few rows of the encoded data
print(data_encoded.head())


   CustomerID   Age  Tenure  Usage Frequency  Support Calls  Payment Delay  \
0         2.0  30.0    39.0             14.0            5.0           18.0   
1         3.0  65.0    49.0              1.0           10.0            8.0   
2         4.0  55.0    14.0              4.0            6.0           18.0   
3         5.0  58.0    38.0             21.0            7.0            7.0   
4         6.0  23.0    32.0             20.0            5.0            8.0   

   Total Spend  Last Interaction  Churn  Gender_Male  \
0        932.0              17.0    1.0        False   
1        557.0               6.0    1.0        False   
2        185.0               3.0    1.0        False   
3        396.0              29.0    1.0         True   
4        617.0              20.0    1.0         True   

   Subscription Type_Premium  Subscription Type_Standard  \
0                      False                        True   
1                      False                       False   
2             

Normalizing and Standardize Numerical Value Coloumn

In [22]:
# Initialize the MinMaxScaler
min_max_scaler = MinMaxScaler()

# Columns to standardize
columns_to_standardize = ['Age', 'Tenure', 'Usage Frequency', 'Total Spend']

# Normalize the specified columns
data_encoded[columns_to_standardize] = min_max_scaler.fit_transform(data_encoded[columns_to_standardize])

# Check the first few rows of the data after normalization
print(data_encoded.head())


   CustomerID       Age    Tenure  Usage Frequency  Support Calls  \
0         2.0  0.255319  0.644068         0.448276            5.0   
1         3.0  1.000000  0.813559         0.000000           10.0   
2         4.0  0.787234  0.220339         0.103448            6.0   
3         5.0  0.851064  0.627119         0.689655            7.0   
4         6.0  0.106383  0.525424         0.655172            5.0   

   Payment Delay  Total Spend  Last Interaction  Churn  Gender_Male  \
0           18.0     0.924444              17.0    1.0        False   
1            8.0     0.507778               6.0    1.0        False   
2           18.0     0.094444               3.0    1.0        False   
3            7.0     0.328889              29.0    1.0         True   
4            8.0     0.574444              20.0    1.0         True   

   Subscription Type_Premium  Subscription Type_Standard  \
0                      False                        True   
1                      False          

In [18]:
# Initialize the MinMaxScaler
min_max_scaler = MinMaxScaler()

# Normalize the specified columns
data[columns_to_standardize] = min_max_scaler.fit_transform(data[columns_to_standardize])

print(data.head())



   CustomerID       Age  Gender    Tenure  Usage Frequency  Support Calls  \
0         2.0  0.255319  Female  0.644068         0.448276            5.0   
1         3.0  1.000000  Female  0.813559         0.000000           10.0   
2         4.0  0.787234  Female  0.220339         0.103448            6.0   
3         5.0  0.851064    Male  0.627119         0.689655            7.0   
4         6.0  0.106383    Male  0.525424         0.655172            5.0   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0           18.0          Standard          Annual     0.924444   
1            8.0             Basic         Monthly     0.507778   
2           18.0             Basic       Quarterly     0.094444   
3            7.0          Standard         Monthly     0.328889   
4            8.0             Basic         Monthly     0.574444   

   Last Interaction  Churn  
0              17.0    1.0  
1               6.0    1.0  
2               3.0    1.0  
3              29.

Feature Engineering


In [23]:
# Assuming 'Last Interaction' is in datetime format, if not, convert it using pd.to_datetime
data_encoded['Last Interaction'] = pd.to_datetime(data_encoded['Last Interaction'])

# Create a new feature: Days since last interaction
data_encoded['Days_since_Last_Interaction'] = (datetime.now() - data_encoded['Last Interaction']).dt.days

# Create Spend per Tenure feature (Total Spend / Tenure)
data_encoded['Spend_per_Tenure'] = data_encoded['Total Spend'] / data_encoded['Tenure']

# Create Spend per Usage feature (Total Spend / Usage Frequency)
data_encoded['Spend_per_Usage'] = data_encoded['Total Spend'] / data_encoded['Usage Frequency']

# Binning Age into groups: Young, Adult, Mature, Senior
data_encoded['Age_group'] = pd.cut(data_encoded['Age'], bins=[-np.inf, 25, 50, 75, np.inf], labels=['Young', 'Adult', 'Mature', 'Senior'])

# Binning Total Spend into categories: Low, Medium, High, Very High
data_encoded['Spend_group'] = pd.qcut(data_encoded['Total Spend'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

# One-hot encode the newly created categorical features: Age_group, Spend_group
data_encoded = pd.get_dummies(data_encoded, columns=['Age_group', 'Spend_group'], drop_first=True)

# Drop original 'Last Interaction' column as it's no longer needed
data_encoded.drop(columns=['Last Interaction'], inplace=True)

# Check the first few rows of the final engineered data
print(data_encoded.head())


   CustomerID       Age    Tenure  Usage Frequency  Support Calls  \
0         2.0  0.255319  0.644068         0.448276            5.0   
1         3.0  1.000000  0.813559         0.000000           10.0   
2         4.0  0.787234  0.220339         0.103448            6.0   
3         5.0  0.851064  0.627119         0.689655            7.0   
4         6.0  0.106383  0.525424         0.655172            5.0   

   Payment Delay  Total Spend  Churn  Gender_Male  Subscription Type_Premium  \
0           18.0     0.924444    1.0        False                      False   
1            8.0     0.507778    1.0        False                      False   
2           18.0     0.094444    1.0        False                      False   
3            7.0     0.328889    1.0         True                      False   
4            8.0     0.574444    1.0         True                      False   

   ...  Contract Length_Quarterly  Days_since_Last_Interaction  \
0  ...                      False     

Model Building

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np


In [29]:
# Remove rows where the target variable 'Churn' is NaN
data_encoded = data_encoded.dropna(subset=['Churn'])



# Now separate the features and target variable
X = data_encoded.drop(columns=['Churn'])
y = data_encoded['Churn']

# Proceed with train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Ensure the shapes are correct
print("Training set shape: ", X_train.shape, y_train.shape)
print("Testing set shape: ", X_test.shape, y_test.shape)


Training set shape:  (308582, 21) (308582,)
Testing set shape:  (132250, 21) (132250,)


Training and Building Logistic Regression


In [33]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Assuming 'data_encoded' is your prepared DataFrame

# Separate features and target variable
X = data_encoded.drop(columns=['Churn'])
y = data_encoded['Churn']

# Replace infinite values with NaN
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Option 1: Drop rows with NaN values
X = X.dropna()
y = y[X.index]  # Ensure y matches the indices of X after dropping NaNs

# Option 2: Impute NaN values (if you choose not to drop them)
# imputer = SimpleImputer(strategy='mean')
# X = imputer.fit_transform(X)

# Optionally, check for extremely large values and clip them
X = np.clip(X, -1e6, 1e6)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [34]:
# Initialize the Logistic Regression model
logreg = LogisticRegression(random_state=42)

# Train the model on the training data
logreg.fit(X_train, y_train)

# Predict on the test data
y_pred = logreg.predict(X_test)
y_pred_proba = logreg.predict_proba(X_test)[:, 1]


In [35]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Print the evaluation metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9762
Precision: 0.9805
Recall: 0.9773
F1 Score: 0.9789
ROC AUC Score: 0.9902
Confusion Matrix:
[[53644  1381]
 [ 1620 69612]]


Now Using The Concept Of Neural Network In The Project

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


In [37]:
# Separate features and target variable
X = data_encoded.drop(columns=['Churn'])
y = data_encoded['Churn']

# Convert target variable to categorical
y = to_categorical(y, num_classes=2)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [38]:
# Initialize the neural network model
model = Sequential()

# Input layer and first hidden layer
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))

# Additional hidden layers
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))

# Output layer
model.add(Dense(2, activation='softmax'))  # Two classes for binary classification

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1)


Epoch 1/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.4336 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 2/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4316 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 3/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.4329 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 4/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4346 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 5/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.4325 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 6/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4328 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 7/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.4320 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 8/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4321 - loss: nan - val_accuracy: 0.4346 - val_loss: nan


In [41]:
# Predict on the test data
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

# Check for NaNs in predictions and probabilities
if np.any(np.isnan(y_pred_proba)):
    print("NaNs found in y_pred_proba")

# Evaluate the model
precision = precision_score(y_test_labels, y_pred, zero_division=0)
recall = recall_score(y_test_labels, y_pred)
f1 = f1_score(y_test_labels, y_pred)

# Ensure y_pred_proba is valid
y_pred_proba = np.nan_to_num(y_pred_proba)

# Check if y_pred_proba has the expected shape
if len(y_pred_proba.shape) == 2 and y_pred_proba.shape[1] == 2:
    roc_auc = roc_auc_score(y_test_labels, y_pred_proba[:, 1])
else:
    roc_auc = None
    print("y_pred_proba shape is not as expected")

# Print the evaluation metrics
print(f"Accuracy: {accuracy_score(y_test_labels, y_pred):.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}" if roc_auc is not None else "ROC AUC Score: Not available")

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test_labels, y_pred))


4133/4133 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step
NaNs found in y_pred_proba
Accuracy: 0.4329
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC AUC Score: 0.5000
Confusion Matrix:
[[57250     0]
 [75000     0]]


In [43]:
#model imbalancing

# Initialize the neural network model with class weights
model = Sequential()
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(2, activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Compute class weights
class_weights = {0: 1.0, 1: 2.0}  # Adjust weights based on class distribution

# Train the model with class weights
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, class_weight=class_weights, verbose=1)


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.4317 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 2/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4328 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 3/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4334 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 4/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.4331 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 5/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4310 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 6/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.4335 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 7/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.4324 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
Epoch 8/20
7715/7715 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.4324 - loss: nan - val_accuracy: 0.4346 - val_loss: nan
